In [1]:
# train_kfold.py
from utils.utils import *
from scripts.config import build_cv_splits, build_model_cfg_from_dataloader  # <- only import helpers
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import StepLR
import wandb
from datetime import datetime

from models.simple_mlp import *


import argparse
import warnings
import os
import time
import json
from typing import Dict, Any, List

try:
    import yaml  # optional, for YAML configs
except Exception:
    yaml = None


warnings.filterwarnings("ignore", message="Accurate seek is not implemented for pyav backend")
torch.manual_seed(0)


In [2]:


def is_number(x):
    try:
        float(x)
        return True
    except Exception:
        return False


def flatten_numeric(d: Dict[str, Any]) -> Dict[str, float]:
    out = {}
    def _walk(prefix, obj):
        if isinstance(obj, dict):
            for k, v in obj.items():
                _walk(f"{prefix}.{k}" if prefix else k, v)
        else:
            if isinstance(obj, (int, float)) or (hasattr(obj, "item") and getattr(obj, "dim", lambda:1)() == 0):
                try:
                    out[prefix] = float(obj if not hasattr(obj, "item") else obj.item())
                except Exception:
                    pass
    _walk("", d)
    return out


def write_json(path: str, obj: Any):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)


def write_csv(path: str, rows: List[Dict[str, Any]]):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    headers = []
    for r in rows:
        for k in r.keys():
            if k not in headers:
                headers.append(k)
    with open(path, "w") as f:
        f.write(",".join(headers) + "\n")
        for r in rows:
            f.write(",".join(str(r.get(h, "")) for h in headers) + "\n")


def load_config_file(path: str) -> Dict[str, Any]:
    ext = os.path.splitext(path)[1].lower()
    with open(path, "r") as f:
        if ext in (".yml", ".yaml"):
            if yaml is None:
                raise RuntimeError("pyyaml not installed; install it or use JSON.")
            return yaml.safe_load(f)
        elif ext == ".json":
            return json.load(f)
        else:
            raise ValueError(f"Unsupported config extension: {ext} (use .yaml/.yml or .json)")


class DefaultArgsNamespace:
    """
    A minimal args container built from a config dict that contains at least:
      - dataloader_params
      - learning_params
      - transformer_params (optional)
      - tcn_model_params (optional)
      - model_cfg_overrides (optional)  # to override d_model, nhead, etc.
    """
    def __init__(self, cfg: Dict[str, Any], fold_index: int = 0):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        # 1) take dataloader params from file (no global imports)
        self.dataloader_params: Dict[str, Any] = dict(cfg["dataloader_params"])

        # 2) build CV folds from the loaded dataloader params
        folds = build_cv_splits(self.dataloader_params)
        if not folds:
            raise RuntimeError("No valid CV folds were produced from the provided dataloader_params['all_trials'] and ['cv'].")

        if fold_index < 0 or fold_index >= len(folds):
            raise IndexError(f"fold_index {fold_index} out of range [0, {len(folds)-1}]")
        self.fold = folds[fold_index]

        # inject chosen fold back
        self.dataloader_params["train_trials"] = self.fold["train_trials"]
        self.dataloader_params["val_trials"]   = self.fold["val_trials"]
        self.dataloader_params["test_trials"]  = self.fold["test_trials"]

        base_exp = self.dataloader_params.get("experiment_name", "exp")
        self.dataloader_params["experiment_name"] = f"{base_exp}_{self.fold['name']}"

        # 3) model cfg derived from dataloader selections (and optional overrides)
        overrides = cfg.get("model_cfg_overrides", {})
        self.mmtransformercfg = build_model_cfg_from_dataloader(self.dataloader_params, **overrides)

        # 4) the rest
        self.learning_params     = dict(cfg["learning_params"])
        self.transformer_params  = dict(cfg.get("transformer_params", {}))
        self.tcn_model_params    = dict(cfg.get("tcn_model_params", {}))
        self.record_results      = bool(cfg.get("record_results", True))



In [3]:


# for testing in notebook: make a mock cmd_args
class MockCmdArgs:
    def __init__(self):
        self.config = "./davinci_configs/MTRSAP/exp_handkp_30hz.yaml"
        self.job_id = None
        self.wandb = "off"
        self.fold_index = None
        
cmd_args = MockCmdArgs()

# load the per-job config (immutable snapshot)
cfg = load_config_file(cmd_args.config)
print("Loaded config:")
print(cfg)

# base job id (stable across folds)
if cmd_args.job_id is None or cmd_args.job_id == "0":
    cmd_args.job_id = str(int(time.time()))

base_job_id = "SANITY_CHECK" + cmd_args.job_id

# build folds *from file config* (for printing)
folds = build_cv_splits(cfg["dataloader_params"])
print(f"Discovered {len(folds)} folds:")
for i, f in enumerate(folds):
    print("-" * 20, f" Fold {i} ", "-" * 20)
    print(f"  [{i}] {f['name']}:\n train (num: {len(f['train_trials'])}) ={f['train_trials']}  \n val (num: {len(f['val_trials'])}) ={f['val_trials']}  \n test (num: {len(f['test_trials'])}) ={f['test_trials']}")
print("-" * 40)



# choose which folds to run
fold_indices = list(range(len(folds))) if cmd_args.fold_index is None else [cmd_args.fold_index]

# top-level run directory (stable across folds)
# collect per-fold metrics for summary
fold_rows_for_csv: List[Dict[str, Any]] = []
numeric_metrics_per_key: Dict[str, List[float]] = {}


Loaded config:
{'record_results': True, 'dataloader_params': {'experiment_name': 'bootcamp_data', 'dataset_name': 'MIDAS', 'base_path': '/standard/UVA-DSA/MIDAS/Organized/Bootcamp/SuturingV2/Processed/', 'batch_size': 1, 'sample_rate': 30, 'ignore_clutch': True, 'clutch_pressed_value': 0, 'drop_neg1': True, 'all_trials': ['S105_T1', 'S106_T1', 'S106_T2', 'S112_T1', 'S112_T2', 'S116_T1', 'S116_T2', 'S116_T4', 'S116_T5', 'S118_T1', 'S200_T1', 'S201_T1', 'S201_T2', 'S202_T1', 'S203_T1', 'S204_T1', 'S209_T2', 'S210_T1', 'S214_T1', 'S214_T4', 'S214_T6', 'S215_T3', 'S215_T4', 'S217_T2', 'S217_T3', 'S217_T4', 'S218_T1', 'S219_T1'], 'cv': {'scheme': 'gesture_flat', 'k': 5, 'val_ratio': 0.25, 'seed': 42, 'shuffle': True, 'val_same_as_test': True}, 'modalities': ['handkp'], 'selections': {'handkp': ['left_index_tip_X', 'left_index_tip_Y', 'left_index_tip_Z', 'right_index_tip_X', 'right_index_tip_Y', 'right_index_tip_Z', 'left_thumb_tip_X', 'left_thumb_tip_Y', 'left_thumb_tip_Z', 'right_thumb_tip

In [4]:

for fi in fold_indices:
    print(f"\n=== Running Fold {fi}: {folds[fi]['name']} ===")
    args = DefaultArgsNamespace(cfg, fold_index=fi)

    modalitys = args.dataloader_params['modalities']
    modality_string = '_'.join(modalitys)
    args.dataloader_params['experiment_name'] = f"{modality_string}_{args.dataloader_params['sample_rate']}hz"
    experiment_name = args.dataloader_params['experiment_name']

    # per-fold folders (unique)
    wandb_mode = "online" if cmd_args.wandb == "on" else "disabled"
    wandb_logger = wandb.init(
        project="MIDAS Gesture Recognition",
        group=f"Gesture Recognition ({base_job_id})",
        mode=wandb_mode,
        name=experiment_name,
        notes=f"job={base_job_id}, fold={folds[fi]['name']}",
        config={"args": str(args.dataloader_params)},  # keep light
    )

    keysteps = args.dataloader_params['keysteps']
    out_classes = len(keysteps)
    modality = args.dataloader_params['modalities']
    selections = args.dataloader_params['selections']
    class_names = None
    class_id_to_name = None

    print(f"Keysteps: {keysteps}")
    print(f"Modalities: {modality}")
    print(f"Selections: {selections}")
    print(f"Trials (train/val/test): {args.dataloader_params['train_trials']} / {args.dataloader_params['val_trials']} / {args.dataloader_params['test_trials']}")

    # Data
    if args.dataloader_params.get('dataset_name', 'MIDAS') == 'MIDAS':
        print("Using MIDAS dataset...")
        train_loader, val_loader, test_loader, train_class_stats, val_class_stats, test_class_stats, class_names = MIDAS_get_dataloaders(args)
        
        # invert class_names so we can go from contiguous index → original_id
        inv_class_names = {v: k for k, v in class_names.items()}

        # build mapping contiguous_index → human-readable name
        class_id_to_name = {
            idx: keysteps.get(orig_id, str(orig_id))
            for idx, orig_id in inv_class_names.items()
        }
        print("Class ID → Name mapping:", class_id_to_name)
    elif args.dataloader_params.get('dataset_name') == 'DESK':
        print("Using DESK dataset...")
        train_loader, val_loader, test_loader, train_class_stats, val_class_stats, test_class_stats = DESK_get_dataloaders(args)
    elif args.dataloader_params.get('dataset_name') == 'JIGSAWS':
        print("Using JIGSAWS dataset...")
        train_loader, val_loader, test_loader, train_class_stats, val_class_stats, test_class_stats = DESK_get_dataloaders(args) # reuse DESK loader for JIGSAWS

    args.dataloader_params['train_class_stats'] = train_class_stats
    args.dataloader_params['val_class_stats'] = val_class_stats

    print(f"Training samples: {len(train_loader.dataset)}, Validation samples: {len(val_loader.dataset)}, Test samples: {len(test_loader.dataset)}")
    print_one_batch(train_loader)

    # Device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    break



=== Running Fold 0: gesture_flat ===
Keysteps: {8: 'Make C Loop', 11: 'Orient Needle', 16: 'Pull Needle out of Tissue', 17: 'Pull Suture', 18: 'Push Needle Through Tissue', 20: 'Reach Suture', 22: 'Square Knot and Cinch', 23: 'Target Needle'}
Modalities: ['handkp']
Selections: {'handkp': ['left_index_tip_X', 'left_index_tip_Y', 'left_index_tip_Z', 'right_index_tip_X', 'right_index_tip_Y', 'right_index_tip_Z', 'left_thumb_tip_X', 'left_thumb_tip_Y', 'left_thumb_tip_Z', 'right_thumb_tip_X', 'right_thumb_tip_Y', 'right_thumb_tip_Z']}
Trials (train/val/test): ['S214_T1', 'S209_T2', 'S200_T1', 'S215_T3', 'S219_T1', 'S116_T2', 'S116_T1', 'S201_T1', 'S203_T1', 'S118_T1', 'S217_T3', 'S201_T2', 'S214_T4', 'S204_T1', 'S217_T4', 'S106_T1', 'S202_T1', 'S106_T2', 'S210_T1', 'S218_T1', 'S112_T2', 'S215_T4', 'S116_T4', 'S116_T5', 'S217_T2', 'S105_T1', 'S112_T1', 'S214_T6'] / [] / ['S214_T1', 'S209_T2', 'S200_T1', 'S215_T3', 'S219_T1', 'S116_T2', 'S116_T1', 'S201_T1', 'S203_T1', 'S118_T1', 'S217_T3',

/sfs/gpfs/tardis/home/cjh9fw/Desktop/2025/DataCollectionSystem/benchmarks/gesture_recognition/MTRSAP/datautils/midas.py:472: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  self.df = self.df.fillna(fillna_value)


Selected classes to allow: {8: 'Make C Loop', 11: 'Orient Needle', 16: 'Pull Needle out of Tissue', 17: 'Pull Suture', 18: 'Push Needle Through Tissue', 20: 'Reach Suture', 22: 'Square Knot and Cinch', 23: 'Target Needle'}
gesture_code dtype before: int64
Filtered rows: 404451 -> 383624
Unique gesture_codes after filtering: ['11', '16', '17', '18', '20', '22', '23', '8']

--- Gesture class mapping ---
Final class_map (gesture_code -> index): {8: 0, 11: 1, 16: 2, 17: 3, 18: 4, 20: 5, 22: 6, 23: 7}
--- Gesture class mapping ---

Dataset will load image features: []
Data split sizes - Train: 1103, Val: 276, Test: 345
Data split ratios - Train: 64.0%, Val: 16.0%, Test: 20.0%

--- Initializing MultimodalGestureDataset ---

Selected classes to allow: {8: 'Make C Loop', 11: 'Orient Needle', 16: 'Pull Needle out of Tissue', 17: 'Pull Suture', 18: 'Push Needle Through Tissue', 20: 'Reach Suture', 22: 'Square Knot and Cinch', 23: 'Target Needle'}


/sfs/gpfs/tardis/home/cjh9fw/Desktop/2025/DataCollectionSystem/benchmarks/gesture_recognition/MTRSAP/datautils/midas.py:472: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  self.df = self.df.fillna(fillna_value)


Selected classes to allow: {8: 'Make C Loop', 11: 'Orient Needle', 16: 'Pull Needle out of Tissue', 17: 'Pull Suture', 18: 'Push Needle Through Tissue', 20: 'Reach Suture', 22: 'Square Knot and Cinch', 23: 'Target Needle'}
gesture_code dtype before: int64
Filtered rows: 404451 -> 383624
Unique gesture_codes after filtering: ['11', '16', '17', '18', '20', '22', '23', '8']

--- Gesture class mapping ---
Final class_map (gesture_code -> index): {8: 0, 11: 1, 16: 2, 17: 3, 18: 4, 20: 5, 22: 6, 23: 7}
--- Gesture class mapping ---

Dataset will load image features: []

--- Initializing MultimodalGestureDataset ---

Selected classes to allow: {8: 'Make C Loop', 11: 'Orient Needle', 16: 'Pull Needle out of Tissue', 17: 'Pull Suture', 18: 'Push Needle Through Tissue', 20: 'Reach Suture', 22: 'Square Knot and Cinch', 23: 'Target Needle'}


/sfs/gpfs/tardis/home/cjh9fw/Desktop/2025/DataCollectionSystem/benchmarks/gesture_recognition/MTRSAP/datautils/midas.py:472: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  self.df = self.df.fillna(fillna_value)


Selected classes to allow: {8: 'Make C Loop', 11: 'Orient Needle', 16: 'Pull Needle out of Tissue', 17: 'Pull Suture', 18: 'Push Needle Through Tissue', 20: 'Reach Suture', 22: 'Square Knot and Cinch', 23: 'Target Needle'}
gesture_code dtype before: int64
Filtered rows: 404451 -> 383624
Unique gesture_codes after filtering: ['11', '16', '17', '18', '20', '22', '23', '8']

--- Gesture class mapping ---
Final class_map (gesture_code -> index): {8: 0, 11: 1, 16: 2, 17: 3, 18: 4, 20: 5, 22: 6, 23: 7}
--- Gesture class mapping ---

Dataset will load image features: []

--- Initializing MultimodalGestureDataset ---

Selected classes to allow: {8: 'Make C Loop', 11: 'Orient Needle', 16: 'Pull Needle out of Tissue', 17: 'Pull Suture', 18: 'Push Needle Through Tissue', 20: 'Reach Suture', 22: 'Square Knot and Cinch', 23: 'Target Needle'}


/sfs/gpfs/tardis/home/cjh9fw/Desktop/2025/DataCollectionSystem/benchmarks/gesture_recognition/MTRSAP/datautils/midas.py:472: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  self.df = self.df.fillna(fillna_value)


Selected classes to allow: {8: 'Make C Loop', 11: 'Orient Needle', 16: 'Pull Needle out of Tissue', 17: 'Pull Suture', 18: 'Push Needle Through Tissue', 20: 'Reach Suture', 22: 'Square Knot and Cinch', 23: 'Target Needle'}
gesture_code dtype before: int64
Filtered rows: 404451 -> 383624
Unique gesture_codes after filtering: ['11', '16', '17', '18', '20', '22', '23', '8']

--- Gesture class mapping ---
Final class_map (gesture_code -> index): {8: 0, 11: 1, 16: 2, 17: 3, 18: 4, 20: 5, 22: 6, 23: 7}
--- Gesture class mapping ---

Dataset will load image features: []
********** ========== **********

Class distribution by split:
Train class stats: {8: 92, 11: 158, 16: 174, 17: 149, 18: 209, 20: 25, 22: 90, 23: 206}
Val   class stats: {8: 23, 11: 39, 16: 44, 17: 37, 18: 53, 20: 6, 22: 23, 23: 51}
Test  class stats: {8: 28, 11: 49, 16: 55, 17: 47, 18: 66, 20: 8, 22: 28, 23: 64}
********** ========== **********
Class ID → Name mapping: {0: 'Make C Loop', 1: 'Orient Needle', 2: 'Pull Needle ou

In [5]:


# for batch in test_loader:
#     print(batch)

print_one_batch(train_loader)
print_one_batch(train_loader)

batch = next(iter(train_loader))
print(batch)


handkp: dtype=torch.float32, shape=(1, 200, 12), min=-0.325, max=0.461
label: dtype=torch.int64, shape=(1,), min=4.000, max=4.000
obs_frame_idx: dtype=torch.int64, shape=(1, 200), min=5157.000, max=5356.000
gesture_code: 1 items -> [18]
trial_id: list of 1 items, first 3: ['S203_T1']
source_csv: list of 1 items, first 3: ['/standard/UVA-DSA/MIDAS/Organized/Bootcamp/SuturingV2/Processed/S203_T1/synched_data/final_annotation_S203_T1.csv']
handkp: dtype=torch.float32, shape=(1, 104, 12), min=-0.325, max=0.461
label: dtype=torch.int64, shape=(1,), min=0.000, max=0.000
obs_frame_idx: dtype=torch.int64, shape=(1, 104), min=26838.000, max=26941.000
gesture_code: 1 items -> [8]
trial_id: list of 1 items, first 3: ['S202_T1']
source_csv: list of 1 items, first 3: ['/standard/UVA-DSA/MIDAS/Organized/Bootcamp/SuturingV2/Processed/S202_T1/synched_data/final_annotation_S202_T1.csv']
{'handkp': tensor([[[ 0.3691,  0.4613, -0.3235,  ...,  0.2618,  0.3766,  0.0009],
         [ 0.3691,  0.4613, -0.3235

In [6]:
# import torch

# def _valid_obs_span(obs_idx_row, mask_row=None):
#     """
#     Returns (start_idx, end_idx) inclusive for a single sample's obs_frame_idx.
#     Handles fixed-length and variable-length (via mask).
#     """
#     if mask_row is not None:
#         # mask_row: [T] bool
#         # valid length = count of True
#         T_valid = int(mask_row.sum().item())
#         if T_valid == 0:
#             return None, None
#         start = int(obs_idx_row[0].item())
#         end   = int(obs_idx_row[T_valid - 1].item())
#         return start, end
#     else:
#         # fixed-length
#         start = int(obs_idx_row[0].item())
#         end   = int(obs_idx_row[-1].item())
#         return start, end

# def _collect_keys(loader, *, verbose_every=0):
#     """
#     Walk a DataLoader and collect:
#       - exact_keys: set of (trial_id, source_csv, start_idx, end_idx)
#       - ranges_by_src: dict[(trial_id, source_csv)] -> list of (start_idx, end_idx)
#     """
#     exact_keys = set()
#     ranges_by_src = {}

#     n = 0
#     for batch in loader:
#         obs = batch["obs_frame_idx"]                # [B, T] or [B, T_max]
#         obs_mask = batch.get("obs_mask", None)      # [B, T_max] (optional)
#         trial_list = batch.get("trial_id", [""]*obs.shape[0])
#         src_list   = batch.get("source_csv", [""]*obs.shape[0])

#         B = obs.shape[0]
#         for i in range(B):
#             trial_id = trial_list[i]
#             source   = src_list[i]
#             obs_row  = obs[i]

#             mask_row = obs_mask[i] if obs_mask is not None else None
#             start, end = _valid_obs_span(obs_row, mask_row)
#             if start is None:
#                 # empty / fully padded sample; skip
#                 continue

#             key = (trial_id, source, start, end)
#             exact_keys.add(key)

#             src_key = (trial_id, source)
#             ranges_by_src.setdefault(src_key, []).append((start, end))

#         n += 1
#         if verbose_every and n % verbose_every == 0:
#             print(f" Scanned {n} batches...")

#     return exact_keys, ranges_by_src

# def _any_temporal_overlap(ranges_a, ranges_b):
#     """
#     Given two lists of (start,end) on the same source, detect if
#     *any* pair overlaps in time.
#     """
#     # Sort for a cheap sweep
#     ra = sorted(ranges_a)
#     rb = sorted(ranges_b)

#     i = j = 0
#     while i < len(ra) and j < len(rb):
#         a0, a1 = ra[i]
#         b0, b1 = rb[j]
#         # Overlap if max(start) <= min(end)
#         if max(a0, b0) <= min(a1, b1):
#             return True, (a0, a1, b0, b1)
#         # advance the one that ends earlier
#         if a1 < b1:
#             i += 1
#         else:
#             j += 1
#     return False, None

# def verify_train_test_disjoint(train_loader, test_loader):
#     print("\n[Verify] Collecting keys from TRAIN...")
#     tr_exact, tr_ranges = _collect_keys(train_loader, verbose_every=0)
#     print("[Verify] Collecting keys from TEST...")
#     te_exact, te_ranges = _collect_keys(test_loader, verbose_every=0)

#     # --- Exact duplicate check ---
#     exact_overlap = tr_exact.intersection(te_exact)
#     if len(exact_overlap) == 0:
#         print("✅ No exact duplicate clips between train and test.")
#     else:
#         print(f"❌ Found {len(exact_overlap)} exact duplicate clips between train and test.")
#         # show up to a few
#         for k in list(exact_overlap)[:10]:
#             trial_id, src, s, e = k
#             print(f"  dup: trial={trial_id} src={src} span=[{s},{e}]")
#         print("  (showing up to 10)")

#     # --- Temporal overlap check (same file, intersecting spans) ---
#     any_overlap = False
#     example = None
#     # Restrict to common (trial,source) keys
#     common_srcs = set(tr_ranges.keys()) & set(te_ranges.keys())
#     for src_key in common_srcs:
#         ov, ex = _any_temporal_overlap(tr_ranges[src_key], te_ranges[src_key])
#         if ov:
#             any_overlap = True
#             example = (src_key, ex)
#             break

#     if not any_overlap:
#         print("✅ No temporal overlap (on the same file) between train and test windows.")
#     else:
#         (trial_id, src), (a0, a1, b0, b1) = example
#         print("⚠️  Temporal overlap detected between train and test windows from the same source.")
#         print(f"   trial={trial_id} src={src}")
#         print(f"   train window [{a0},{a1}] overlaps test window [{b0},{b1}].")
#         print("   (This happens when you split at the window level with overlapping windows.)")

#     # --- Internal duplicates (optional) ---
#     def count_dupes(keys):
#         # count identical keys within the same split
#         from collections import Counter
#         c = Counter(keys)
#         return sum(1 for k,v in c.items() if v > 1)

#     n_dupe_train = count_dupes(tr_exact)
#     n_dupe_test  = count_dupes(te_exact)
#     if n_dupe_train == 0 and n_dupe_test == 0:
#         print("✅ No duplicates within train or within test.")
#     else:
#         if n_dupe_train:
#             print(f"⚠️  {n_dupe_train} duplicate clip keys within TRAIN.")
#         if n_dupe_test:
#             print(f"⚠️  {n_dupe_test} duplicate clip keys within TEST.")


# verify_train_test_disjoint(train_loader, test_loader)

